In [1]:
import pandas as pd
import numpy as np
from pvlib.location import Location
from pvlib.irradiance import get_total_irradiance

In [4]:
# 1. CONFIGURATION
#  Location (Munich Campus)
lat, lon = 48.14904043542013, 11.567681331388638
altitude = 519 

surface_tilt = 30 
surface_azimuth = 180  # (south)

# Define the full year time index (1-minute resolution)
times = pd.date_range(start='2013-01-01 00:00:00', end='2013-12-31 23:59:00', freq='1min')

# 2. GENERATE REFERENCE CLEAR SKY POA
site = Location(lat, lon, altitude=altitude, name='Munich Campus')

# Calcultae solar position
solar_position = site.get_solarposition(times)

# Calculate Clear Sky GHI/DNI/DHI
cs = site.get_clearsky(times)

# Transpose Clear Sky Data to Plane of Array (POA)
cs_poa = get_total_irradiance(
    surface_tilt,
    surface_azimuth,
    solar_position['apparent_zenith'],
    solar_position['azimuth'],
    cs['dni'],
    cs['ghi'],
    cs['dhi']
)

# We only care about the total radiation 
reference_poa = cs_poa['poa_global']

# 3. PROCESS USER DATA
df_weather_12h = pd.read_csv(f'Data/Munich_weather_12h.csv')
df_weather_12h = df_weather_12h.drop(columns=['Unnamed: 0'])
df_weather_12h['timestamp'] = pd.date_range(start='2013-01-01 00:00:00', periods=len(df_weather_12h), freq='12h')
df_weather_12h = df_weather_12h.set_index('timestamp')

# 4. DOWNSCALING RADIATION (Stochastic Method)

# Resample reference to 12h to macht input data
ref_12h = reference_poa.resample('12h').mean()

# Calculate Clearness Index
df_combined = pd.concat([df_weather_12h['G_Gk'], ref_12h], axis=1)

# If Reference is 0, Kt is 0
df_combined['Kt'] = df_combined['G_Gk'] / df_combined['poa_global'].replace(0, np.nan)
df_combined['Kt'] = df_combined['Kt'].fillna(0).clip(0, 1.2)

# Upsample Kt to 1-minute resolution
Kt_1min = df_combined['Kt'].reindex(times).ffill()

# Apply Base Calculation (Reference * Kt)
poa_1min = reference_poa * Kt_1min

# Add Stochastic Noise (Clouds)
daylight_mask = (reference_poa > 10)
cloudy_mask = (Kt_1min > 0.2) & (Kt_1min < 0.8)

# Generate noise
noise_scale = 0.3 * (1 - abs(Kt_1min - 0.5) * 2)
random_noise = np.random.normal(0, 1, size=len(times))

# Smooth the noise (clouds dont flicker instantly every minute)
smoothed_noise = pd.Series(random_noise).rolling(20, min_periods=1, center=True).mean()

# Apply noise 
final_poa = poa_1min.copy()
final_poa[daylight_mask & cloudy_mask] *= (1 + smoothed_noise[daylight_mask & cloudy_mask] * noise_scale[daylight_mask & cloudy_mask])
# Final Cleanup
final_poa = final_poa.clip(lower=0)

# Since Temp doesn't jump like light, we use PCHIP interpolation
# (Piecewise Cubic Hermite Interpolating Polynomial) - it's monotonic and prevents overshooting
temp_1min = df_weather_12h['Ta'].reindex(times).interpolate(method='pchip')

# Add slight thermal jitter (approx +/- 0.2 C) to look realistic
temp_noise = np.random.normal(0, 0.1, len(times))
temp_1min += temp_noise

# --- 6. EXPORT ---
df_result = pd.DataFrame({
    'POA_Global_Wm2': final_poa,
    'T_Ambient_C': temp_1min
}, index=times)

print("Done! First 5 rows:")
print(df_result.head())


IndexingError: Unalignable boolean Series provided as indexer (index of the boolean Series and of the indexed object do not match).

In [ ]:
# Input 